# Tier-2: multi-task sequence CNN for CRISPR element→gene regulation (K562)

**The Go/No-Go.** The Tier-1 gradient-boosted model reached **AUPRC 0.608** using the *measured* ChIP of 311 TFs (identity, not count), vs **0.519** epigenetics-only and **0.393** distance — all chromosome-held-out. This notebook trains a deep **multi-task sequence CNN** that must earn its keep:

- element DNA (600bp, one-hot) → conv **motif encoder** → embedding
- **aux head**: predict which of 311 TFs bind (dense, ~1.2M labels) — forces the encoder to *learn the TF motif grammar from sequence*
- **main head**: `[embedding + 8 tabular features]` → regulation (CRISPR `Significant`, 569 positives)

**Honest bars:** beating 0.608 is hard (that GBM *measures* binding; this model must *predict* it from DNA). The real win is (a) beat **epigenetics-only 0.519** → it learned TF grammar from sequence; and (b) approach 0.608 **without ChIP at inference** → a portable predictor. Evaluated chromosome-held-out, AUPRC, 3 seeds + a **label-shuffle control** (must collapse to base rate ~0.055).

All data is real ENCODE K562 + the ENCODE CRISPR benchmark, already committed to the repo. **Set Runtime → Change runtime type → GPU** before running.

In [ ]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU  <-- set Runtime > Change runtime type > GPU for a fast run')

In [ ]:
# Clone the repo (all data + code committed). For a PRIVATE repo, set TOKEN below.
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
TOKEN  = ''                     # <-- if private: a GitHub personal-access token with repo read
OWNER, REPO = 'nikku03', 'cell'
url = f'https://{TOKEN + "@" if TOKEN else ""}github.com/{OWNER}/{REPO}.git'
import os
if not os.path.isdir(REPO):
    !git clone --depth 1 --branch $BRANCH $url
%cd $REPO
!pip -q install xgboost
import sys; sys.path.insert(0, 'colab')
# sanity: the three data files the models need
for f in ['outputs/orphan/invivo/element_seqs.json','outputs/orphan/invivo/compendium_tf.json','outputs/orphan/crispr_features_compendium.csv']:
    print(('OK ' if os.path.exists(f) else 'MISSING '), f)

## 1. Tier-1 baselines (GBM) — same chromosome-held-out protocol
Reproduces distance / epigenetics-only / TF-identity(311 measured ChIP) AUPRC + label-shuffle control, so the deep model is compared apples-to-apples in this environment.

In [ ]:
import crispr_gate as g
I = g.identity_test()
print('Tier-1 GBM baselines (AUPRC, 3 seeds, chromosome-held-out):')
for k in ['distance','epigenetics','epi+TF_identity_311','shuffled_control']:
    print(f'  {k:22s} {I[k]["auprc"]:.3f}   seeds {I[k]["seeds"]}')
print(f'  base rate {I["base_rate"]}')

## 2. Tier-2 multi-task sequence CNN (GPU)
Trains 3 seeds + a label-shuffle control, chromosome-held-out. On a Colab GPU this is a few minutes; each fold prints its AUPRC as it completes.

In [ ]:
import seq_model as m
out = m.main()

## 3. Verdict
The `main()` call above prints the verdict. Summary of what a good result looks like:

| result | meaning |
|---|---|
| seq CNN ≥ 0.608 | matches/beats measured-ChIP GBM **without ChIP** — strong adopt |
| seq CNN > 0.519 | learned TF grammar from sequence, ChIP-free — adopt |
| seq CNN ≈ 0.519 | ties epigenetics; ship the Tier-1 GBM |
| seq CNN < 0.519 | too few positives for the CNN; ship the GBM |
| shuffle control ≈ 0.055 | confirms no leakage/overfit (must hold) |

Whatever it lands on is the honest answer — the deep model is adopted **only if it earns it** on held-out chromosomes.